# Tokenizer Experiments

just need something simple for testing inference stuff
not trying to build a full tokenizer like HuggingFace

In [42]:
# simple word tokenizer
class SimpleTokenizer:
    def __init__(self, vocab):
        self.vocab = vocab
        self.inv_vocab = {i: t for t, i in vocab.items()}

    def encode(self, text):
        return [self.vocab.get(tok, self.vocab['<UNK>']) for tok in text.split()]

    def decode(self, token_ids):
        return ' '.join([self.inv_vocab.get(i, '<UNK>') for i in token_ids])

In [43]:
# test vocab
vocab = {
    '<PAD>': 0,
    '<UNK>': 1,
    'the': 2,
    'cat': 3,
    'sat': 4,
    'on': 5,
    'mat': 6,
}

tokenizer = SimpleTokenizer(vocab)
tokenizer.encode("the cat sat on the mat")

[2, 3, 4, 5, 2, 6]

In [44]:
# decode it back
ids = [2, 3, 4, 5, 2, 6]
tokenizer.decode(ids)

'the cat sat on the mat'

In [45]:
# what about unknown words?
tokenizer.encode("the dog ran")

[2, 1, 1]

### char level version

might be easier - no unknown tokens to deal with

In [46]:
class CharTokenizer:
    def __init__(self):
        chars = ' abcdefghijklmnopqrstuvwxyz'
        self.vocab = {c: i for i, c in enumerate(chars)}
        self.inv_vocab = {i: c for c, i in self.vocab.items()}
    
    def encode(self, text):
        return [self.vocab.get(c, 0) for c in text.lower()]
    
    def decode(self, ids):
        return ''.join([self.inv_vocab.get(i, ' ') for i in ids])

In [47]:
char_tok = CharTokenizer()
ids = char_tok.encode("hello world")
print(ids)
print(char_tok.decode(ids))

[8, 5, 12, 12, 15, 0, 23, 15, 18, 12, 4]
hello world


### build vocab from text

In [48]:
from collections import Counter

def build_vocab(text, vocab_size=50):
    words = text.lower().split()
    counts = Counter(words)
    
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in counts.most_common(vocab_size - 2):
        vocab[word] = len(vocab)
    
    return vocab

In [49]:
# try it
text = "the quick brown fox jumps over the lazy dog the dog was lazy"
vocab = build_vocab(text)
print(vocab)

{'<PAD>': 0, '<UNK>': 1, 'the': 2, 'lazy': 3, 'dog': 4, 'quick': 5, 'brown': 6, 'fox': 7, 'jumps': 8, 'over': 9, 'was': 10}


In [50]:
tok = SimpleTokenizer(vocab)
tok.encode("the dog jumps")

[2, 4, 8]

### integrate with model from nb 00

In [51]:
import torch

# make bigger vocab
corpus = "the cat sat on the mat the dog ran fast a quick brown fox jumps"
vocab = build_vocab(corpus, vocab_size=100)

tokenizer = SimpleTokenizer(vocab)

# test encoding
prompt = "the cat sat"
ids = tokenizer.encode(prompt)
print(f"text: {prompt}")
print(f"ids: {ids}")

# these ids can go straight to model.forward(torch.tensor([ids])) in nb 00
# and decode output with tokenizer.decode(output_ids)

text: the cat sat
ids: [2, 3, 4]


done - tokenizer is ready to use with the GPT model